# EXP-005 — Minimal Source-BFS Baseline

Purpose: create a clean, reproducible ARC-AGI-3 baseline that tests the strongest idea observed in public notebooks:

> locate the current game source, instantiate the game offline, run a bounded BFS planner, then replay the discovered solution online.

This notebook intentionally keeps the first version small:
- no CNN fallback
- no hidden-state hash yet
- no level-transfer yet
- simple visible-pixel fallback when BFS fails

Current comparison target from `docs/experiment_tracker.md`:
- EXP-003B public score: `0.12`
- EXP-003B local score: `0.4849109618`
- EXP-003B levels: `6 / 183`

Validation gate:
- pass Kaggle commit-mode mechanics
- produce `submission.parquet`
- log BFS source path, scanned actions, explored states, solution length, and fallback usage
- do not claim improvement until local/Kaggle results are recorded


In [ ]:
!pip install -q --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py

# ================================================================
# EXP-005 — Minimal Source-BFS Baseline
#
# Clean first implementation:
# 1) find current game source
# 2) import the game class
# 3) instantiate offline simulator
# 4) scan effective actions
# 5) bounded BFS
# 6) replay if solved
# 7) simple visible-pixel fallback if not solved
#
# Deliberately NOT included yet:
# - CNN fallback
# - hidden-state hashing
# - solution transfer
# - counter/A*/IDDFS variants
# ================================================================
import copy
import glob
import hashlib
import importlib.util
import json
import logging
import os
import random
import time
import traceback
from collections import Counter, defaultdict, deque
from pathlib import Path

import numpy as np

from agents.agent import Agent
from arcengine import ActionInput, GameAction, GameState

logger = logging.getLogger(__name__)


def _action_id(a):
    """Normalize action enum/int to int id."""
    try:
        return int(a.value)
    except Exception:
        return int(a)


def _last_frame_array(result_or_fd):
    """Return the latest 64x64 frame as numpy int array."""
    frame = getattr(result_or_fd, "frame", None)
    if frame is None:
        return None
    arr = np.asarray(frame)
    if arr.ndim == 3:
        return arr[-1].astype(np.int64)
    if arr.ndim == 2:
        return arr.astype(np.int64)
    return None


def _make_action(action_id, data=None):
    action = GameAction.from_id(int(action_id))
    if data:
        action.set_data(dict(data))
    return action


def _make_action_input(action_id, data=None):
    gid = GameAction.from_id(int(action_id))
    if data:
        return ActionInput(id=gid, data=dict(data))
    return ActionInput(id=gid)


def find_game_source_and_class(game_id, arc_env=None):
    """
    Prefer the official environment local_dir, then fall back to conservative glob search.
    Returns (source_path, class_name) or (None, guessed_class_name).
    """
    gid = str(game_id).split("-")[0]
    class_guess = gid.capitalize()
    if len(gid) == 4 and gid[0].isalpha():
        class_guess = gid[0].upper() + gid[1:]

    def class_from_source(path, fallback):
        try:
            import re
            content = Path(path).read_text(errors="ignore")[:5000]
            m = re.search(r"class\\s+(\\w+)\\s*\\(\\s*ARCBaseGame", content)
            return m.group(1) if m else fallback
        except Exception:
            return fallback

    # Best path: current environment metadata
    try:
        info = getattr(arc_env, "environment_info", None)
        local_dir = getattr(info, "local_dir", None)
        if local_dir:
            root = Path(local_dir)
            candidates = [
                root / f"{gid}.py",
                root / f"{gid.lower()}.py",
                root / f"{class_guess.lower()}.py",
            ]
            candidates += list(root.rglob(f"{gid}.py"))[:10]
            candidates += list(root.rglob(f"{gid.lower()}.py"))[:10]
            for cand in candidates:
                if cand.exists():
                    return str(cand), class_from_source(cand, class_guess)
    except Exception:
        pass

    # Fallback search patterns; intentionally limited.
    for pattern in [
        f"/tmp/**/{gid}.py",
        f"/kaggle/working/**/{gid}.py",
        f"/kaggle/input/**/{gid}.py",
        f"**/game_sources/**/{gid}.py",
    ]:
        try:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                src = matches[0]
                return src, class_from_source(src, class_guess)
        except Exception:
            continue

    return None, class_guess


class MinimalBFSSolver:
    """Small, bounded BFS planner over a directly instantiated game class."""

    def __init__(self, game_path, class_name, scan_timeout=4.0, bfs_timeout=25.0,
                 max_states=50000, max_depth=30):
        self.game_path = game_path
        self.class_name = class_name
        self.scan_timeout = float(scan_timeout)
        self.bfs_timeout = float(bfs_timeout)
        self.max_states = int(max_states)
        self.max_depth = int(max_depth)
        self.game_cls = None
        self.last_log = {}

    def load(self):
        try:
            spec = importlib.util.spec_from_file_location("exp005_game_module", self.game_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            self.game_cls = getattr(module, self.class_name)
            return True
        except Exception as exc:
            self.last_log = {"load_error": repr(exc), "game_path": self.game_path, "class_name": self.class_name}
            logger.warning("EXP005 BFS load failed: %s", exc)
            return False

    @staticmethod
    def _frame_hash(frame):
        return hashlib.md5(np.asarray(frame, dtype=np.uint8).tobytes()).hexdigest()

    def _reset_level(self, level_idx):
        game = self.game_cls()
        game.set_level(int(level_idx))
        game.perform_action(_make_action_input(GameAction.RESET.value), raw=True)
        result = game.perform_action(_make_action_input(GameAction.RESET.value), raw=True)
        frame = _last_frame_array(result)
        if frame is None:
            frame = np.asarray(game.get_pixels(0, 0, 64, 64), dtype=np.int64)
        return game, frame

    def _available_ids(self, game):
        raw = getattr(game, "_available_actions", [])
        ids = []
        for a in raw:
            try:
                aid = _action_id(a)
                if aid not in ids:
                    ids.append(aid)
            except Exception:
                pass
        return ids

    def _scan_actions(self, game, start_frame):
        """Return list of (action_id, data) that change the frame from start_frame."""
        actions = []
        start_frame = np.asarray(start_frame)
        bg = int(np.bincount(start_frame.flatten(), minlength=16).argmax())
        available = self._available_ids(game)

        # Directional / interact actions.
        for aid in [a for a in available if 1 <= a <= 5]:
            try:
                g = copy.deepcopy(game)
                result = g.perform_action(_make_action_input(aid), raw=True)
                frame = _last_frame_array(result)
                if frame is not None and int(np.sum(start_frame != frame)) > 0:
                    actions.append((aid, None))
            except Exception:
                continue

        # Click action scan over non-background pixels.
        if 6 in available:
            t0 = time.time()
            seen_effects = set()
            hit_positions = []
            for y in range(0, 64, 2):
                if time.time() - t0 > self.scan_timeout:
                    break
                for x in range(0, 64, 2):
                    if start_frame[y, x] == bg:
                        continue
                    try:
                        data = {"x": int(x), "y": int(y), "game_id": "exp005_bfs"}
                        g = copy.deepcopy(game)
                        result = g.perform_action(_make_action_input(6, data), raw=True)
                        frame = _last_frame_array(result)
                        if frame is None:
                            continue
                        if int(np.sum(start_frame != frame)) > 0:
                            effect = self._frame_hash(frame)
                            if effect not in seen_effects:
                                seen_effects.add(effect)
                                actions.append((6, data))
                                hit_positions.append((x, y))
                    except Exception:
                        continue

            # Neighbor refinement around effective stride-2 clicks.
            tried = set(hit_positions)
            for hx, hy in hit_positions[:128]:
                if time.time() - t0 > self.scan_timeout * 1.5:
                    break
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = hx + dx, hy + dy
                    if not (0 <= nx < 64 and 0 <= ny < 64) or (nx, ny) in tried:
                        continue
                    tried.add((nx, ny))
                    if start_frame[ny, nx] == bg:
                        continue
                    try:
                        data = {"x": int(nx), "y": int(ny), "game_id": "exp005_bfs"}
                        g = copy.deepcopy(game)
                        result = g.perform_action(_make_action_input(6, data), raw=True)
                        frame = _last_frame_array(result)
                        if frame is None:
                            continue
                        if int(np.sum(start_frame != frame)) > 0:
                            effect = self._frame_hash(frame)
                            if effect not in seen_effects:
                                seen_effects.add(effect)
                                actions.append((6, data))
                    except Exception:
                        continue

        return actions

    def solve_level(self, level_idx):
        """Bounded BFS. Returns list[(action_id, data)] or None."""
        log = {
            "level_idx": int(level_idx),
            "game_path": self.game_path,
            "class_name": self.class_name,
            "loaded": self.game_cls is not None,
            "scanned_actions": 0,
            "explored": 0,
            "unique_states": 0,
            "solution_len": None,
            "status": "started",
            "error": None,
        }
        t0 = time.time()

        try:
            base_game, start_frame = self._reset_level(level_idx)
            actions = self._scan_actions(base_game, start_frame)
            log["scanned_actions"] = len(actions)

            if not actions:
                log["status"] = "no_effective_actions"
                self.last_log = log
                return None

            visited = {self._frame_hash(start_frame)}
            queue = deque([(copy.deepcopy(base_game), [], 0)])

            while queue and log["explored"] < self.max_states and (time.time() - t0) < self.bfs_timeout:
                game, hist, depth = queue.popleft()

                for aid, data in actions:
                    if log["explored"] >= self.max_states or (time.time() - t0) >= self.bfs_timeout:
                        break

                    try:
                        g2 = copy.deepcopy(game)
                        result = g2.perform_action(_make_action_input(aid, data), raw=True)
                        log["explored"] += 1

                        frame = _last_frame_array(result)
                        if frame is None:
                            continue

                        h = self._frame_hash(frame)
                        if h in visited:
                            continue
                        visited.add(h)

                        new_hist = hist + [(aid, data)]

                        current_level = getattr(g2, "_current_level_index", level_idx)
                        result_completed = getattr(result, "levels_completed", level_idx)
                        if result_completed > level_idx or current_level > level_idx:
                            log["unique_states"] = len(visited)
                            log["solution_len"] = len(new_hist)
                            log["status"] = "solved"
                            log["elapsed_sec"] = round(time.time() - t0, 3)
                            self.last_log = log
                            return new_hist

                        if depth + 1 < self.max_depth:
                            queue.append((g2, new_hist, depth + 1))
                    except Exception:
                        continue

            log["unique_states"] = len(visited)
            log["elapsed_sec"] = round(time.time() - t0, 3)
            log["status"] = "timeout_or_exhausted"
            self.last_log = log
            return None

        except Exception as exc:
            log["status"] = "error"
            log["error"] = repr(exc)
            log["elapsed_sec"] = round(time.time() - t0, 3)
            self.last_log = log
            logger.warning("EXP005 BFS solve error: %s", exc)
            return None


class MyAgent(Agent):
    """Planner-first agent with a simple visible-pixel fallback."""

    MAX_ACTIONS = float("inf")
    _MAX_FRAMES = 10

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1e6) ^ (hash(getattr(self, "game_id", "unknown")) & 0xFFFFFFFF)
        self.rng = random.Random(seed)
        np.random.seed(seed % (2**32 - 1))

        self.current_level = None
        self.bfs = None
        self.bfs_ready = False
        self.bfs_source_path = None
        self.bfs_class_name = None
        self.solution = None
        self.solution_step = 0
        self.tried_bfs_levels = set()

        self.prev_frame = None
        self.prev_level = None
        self.prev_action_id = None
        self.action_stats = defaultdict(lambda: {"n": 0, "progress": 0, "noop": 0, "game_over": 0})
        self.action_counts = Counter()
        self.policy_counts = Counter()
        self.bfs_logs = []

        self.run_start = time.time()
        self.max_total_runtime = 8 * 3600 - 300

    def append_frame(self, f):
        self.frames.append(f)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if getattr(f, "guid", None):
            self.guid = f.guid
        if hasattr(self, "recorder") and not self.is_playback:
            try:
                self.recorder.record(json.loads(f.model_dump_json()))
            except Exception:
                pass

    def _level(self, lf):
        # Prefer levels_completed. Some historical notebooks used score, but
        # level index semantics are cleaner for planner handoff.
        return int(getattr(lf, "levels_completed", 0) or 0)

    def _raw(self, lf):
        frame = _last_frame_array(lf)
        if frame is None:
            return np.zeros((64, 64), dtype=np.int64)
        return frame

    def _available_ids(self, lf):
        avail = getattr(lf, "available_actions", None) or []
        ids = []
        for a in avail:
            try:
                aid = _action_id(a)
                if aid not in ids:
                    ids.append(aid)
            except Exception:
                pass
        return ids

    def _maybe_update_stats(self, raw, level, state):
        if self.prev_frame is None or self.prev_action_id is None:
            return

        stat = self.action_stats[int(self.prev_action_id)]
        stat["n"] += 1

        if level > (self.prev_level or 0):
            stat["progress"] += 1

        if np.array_equal(raw, self.prev_frame):
            stat["noop"] += 1

        if state is GameState.GAME_OVER:
            stat["game_over"] += 1

    def _init_bfs_once(self):
        if self.bfs_ready:
            return

        src, cls_name = find_game_source_and_class(self.game_id, getattr(self, "arc_env", None))
        self.bfs_source_path = src
        self.bfs_class_name = cls_name

        if src:
            self.bfs = MinimalBFSSolver(
                src,
                cls_name,
                scan_timeout=float(os.getenv("EXP005_SCAN_TIMEOUT", "4")),
                bfs_timeout=float(os.getenv("EXP005_BFS_TIMEOUT", "25")),
                max_states=int(os.getenv("EXP005_MAX_STATES", "50000")),
                max_depth=int(os.getenv("EXP005_MAX_DEPTH", "30")),
            )
            if not self.bfs.load():
                self.bfs = None

        self.bfs_ready = True

    def _try_bfs_for_level(self, level):
        if level in self.tried_bfs_levels:
            return

        self.tried_bfs_levels.add(level)
        self._init_bfs_once()

        if self.bfs is None:
            self.bfs_logs.append({
                "level_idx": level,
                "status": "source_not_found_or_load_failed",
                "game_path": self.bfs_source_path,
                "class_name": self.bfs_class_name,
            })
            return

        sol = self.bfs.solve_level(level)
        self.bfs_logs.append(dict(self.bfs.last_log))

        if sol:
            self.solution = sol
            self.solution_step = 0

    def _fallback_action(self, raw, lf):
        available = self._available_ids(lf)
        available = [a for a in available if a in [1, 2, 3, 4, 5, 6, 7]]

        # Small online action prior based on progress/no-op/game-over aggregate.
        usable_stats = []
        for aid in available:
            if aid == 7:
                continue
            st = self.action_stats.get(aid)
            if not st or st["n"] < 8:
                continue
            score = (5.0 * st["progress"] - 1.5 * st["game_over"] - 0.15 * st["noop"]) / max(st["n"], 1)
            usable_stats.append((score, aid))

        if usable_stats and self.rng.random() < 0.10:
            usable_stats.sort(reverse=True)
            aid = usable_stats[0][1]
            if aid == 6:
                click = self._random_visible_click(raw)
                if click:
                    action = GameAction.ACTION6
                    action.set_data(click)
                    action.reasoning = "fallback:prior_click"
                    return action, 6
            else:
                action = _make_action(aid)
                action.reasoning = "fallback:prior_action"
                return action, aid

        # Visible-pixel click fallback.
        if 6 in available and self.rng.random() < 0.65:
            click = self._random_visible_click(raw)
            if click:
                action = GameAction.ACTION6
                action.set_data(click)
                action.reasoning = "fallback:visible_click"
                return action, 6

        move_ids = [a for a in available if 1 <= a <= 5]
        if move_ids:
            aid = self.rng.choice(move_ids)
            action = _make_action(aid)
            action.reasoning = "fallback:random_move"
            return action, aid

        if 7 in available:
            action = _make_action(7)
            action.reasoning = "fallback:undo_last_resort"
            return action, 7

        action = GameAction.RESET
        action.reasoning = "fallback:reset_last_resort"
        return action, 0

    def _random_visible_click(self, raw):
        try:
            counts = np.bincount(raw.flatten(), minlength=16)
            bg = int(counts.argmax())
            ys, xs = np.where(raw != bg)
            if len(xs) == 0:
                return None
            idx = self.rng.randrange(len(xs))
            return {"x": int(xs[idx]), "y": int(ys[idx]), "game_id": "exp005_fallback"}
        except Exception:
            return None

    def is_done(self, frames, latest_frame):
        try:
            return latest_frame.state is GameState.WIN or (time.time() - self.run_start) >= self.max_total_runtime
        except Exception:
            return True

    def choose_action(self, frames, latest_frame):
        try:
            level = self._level(latest_frame)
            raw = self._raw(latest_frame)
            state = getattr(latest_frame, "state", None)

            self._maybe_update_stats(raw, level, state)

            if state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self.prev_frame = None
                self.prev_action_id = None
                self.prev_level = level
                self.policy_counts["reset"] += 1
                action = GameAction.RESET
                action.reasoning = "reset"
                return action

            if level != self.current_level:
                self.current_level = level
                self.solution = None
                self.solution_step = 0
                self._try_bfs_for_level(level)

            if self.solution and self.solution_step < len(self.solution):
                aid, data = self.solution[self.solution_step]
                self.solution_step += 1
                action = _make_action(aid, data)
                action.reasoning = f"bfs_replay:{self.solution_step}/{len(self.solution)}"
                self.policy_counts["bfs_replay"] += 1
                self.action_counts[int(aid)] += 1
                self.prev_frame = raw.copy()
                self.prev_level = level
                self.prev_action_id = int(aid)
                return action

            action, aid = self._fallback_action(raw, latest_frame)
            self.policy_counts["fallback"] += 1
            self.action_counts[int(aid)] += 1
            self.prev_frame = raw.copy()
            self.prev_level = level
            self.prev_action_id = int(aid)
            return action

        except Exception as exc:
            traceback.print_exc()
            self.policy_counts["error_fallback"] += 1
            action = random.choice([
                GameAction.ACTION1,
                GameAction.ACTION2,
                GameAction.ACTION3,
                GameAction.ACTION4,
                GameAction.ACTION5,
            ])
            action.reasoning = f"error_fallback:{exc}"
            return action


In [ ]:

import os
import shutil
import subprocess
from pathlib import Path

if os.getenv("KAGGLE_IS_COMPETITION_RERUN") == "1":
    BASE = Path("/kaggle/working/ARC-AGI-3-Agents")
    SRC = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    AGENT_SRC = Path("/kaggle/working/my_agent.py")
    AGENT_DST = BASE / "agents/templates/my_agent.py"

    # Gateway handshake.
    subprocess.run(
        [
            "curl",
            "--fail",
            "--retry", "60",
            "--retry-all-errors",
            "--retry-delay", "5",
            "--retry-max-time", "300",
            "http://gateway:8001/api/games",
        ],
        check=False,
    )

    if BASE.exists():
        shutil.rmtree(BASE)
    shutil.copytree(SRC, BASE)

    if not AGENT_SRC.exists():
        raise FileNotFoundError("my_agent.py not found")
    shutil.copy(AGENT_SRC, AGENT_DST)

    (BASE / "agents/__init__.py").write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.my_agent import MyAgent\n\n"
        "load_dotenv()\n\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        '    "random": Random,\n'
        '    "myagent": MyAgent,\n'
        "}\n"
    )

    (BASE / ".env").write_text(
        "\n".join(
            [
                "SCHEME=http",
                "HOST=gateway",
                "PORT=8001",
                "ARC_API_KEY=test-key-123",
                "ARC_BASE_URL=http://gateway:8001/",
                "OPERATION_MODE=online",
                "RECORDINGS_DIR=/kaggle/working/server_recording",
            ]
        )
        + "\n"
    )

    subprocess.run(
        ["python", "main.py", "--agent", "myagent"],
        cwd=str(BASE),
        env={**os.environ, "MPLBACKEND": "agg", "PYTHONUNBUFFERED": "1"},
        check=True,
    )
else:
    print("Local/non-rerun mode: competition gateway is not available. Writing dummy submission fallback.")


In [ ]:

import os

if os.getenv("KAGGLE_IS_COMPETITION_RERUN") != "1":
    import pandas as pd

    submission = pd.DataFrame(
        [
            {
                "row_id": "debug_0",
                "game_id": "1",
                "end_of_game": True,
                "score": 0,
            }
        ]
    )
    submission.to_parquet("/kaggle/working/submission.parquet", index=False)
    print("Wrote /kaggle/working/submission.parquet dummy fallback.")


## Tracker row draft

Do not add this to `docs/experiment_tracker.md` until the notebook has a real local or Kaggle result.

```text
| EXP-005 | 2026-05-15 | `notebooks/01_exploration/exp005_minimal_source_bfs_baseline.ipynb` | Minimal source-BFS + visible-pixel fallback | First clean source-code BFS reproduction: source lookup, direct game instantiation, effective-action scan, bounded BFS, online replay, simple fallback | Pending | Pending | Created / awaiting local validation | Gate: must beat EXP-003B local score or produce clear BFS-solved-level logs. |
```
